In [ ]:
# Clone the github repository
!git clone https://github.com/Cesco16/Nanopore_Sequencing_Lab.git

Cloning into 'Nanopore_Sequencing_Lab'...
remote: Enumerating objects: 43, done.
remote: Counting objects: 100% (43/43), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 43 (delta 19), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (43/43), 5.40 MiB | 15.20 MiB/s, done.
Resolving deltas: 100% (19/19), done.


In [ ]:
# Move to the cloned repository
%cd Nanopore_Sequencing_Lab
!ls -la

/content/Nanopore_Sequencing_Lab
total 5972
drwxr-xr-x 3 root root    4096 Jun 22 08:32 .
drwxr-xr-x 1 root root    4096 Jun 22 08:35 ..
-rw-r--r-- 1 root root   21477 Jun 22 08:32 condacolab_install.log
-rw-r--r-- 1 root root     145 Jun 22 08:32 forensic.yaml
drwxr-xr-x 8 root root    4096 Jun 22 08:32 .git
-rw-r--r-- 1 root root     106 Jun 22 08:32 GWAS.yaml
-rw-r--r-- 1 root root    1074 Jun 22 08:32 LICENSE
-rw-r--r-- 1 root root 1270492 Jun 22 08:32 nanosequencinglab.png
-rw-r--r-- 1 root root 4783699 Jun 22 08:32 practical-GWAS.pdf
-rw-r--r-- 1 root root     302 Jun 22 08:32 README.md
-rw-r--r-- 1 root root     198 Jun 22 08:32 SV.yaml


In [ ]:
# Istall condacolab
!pip install -q condacolab
import condacolab
condacolab.install()

⏬ Downloading https://github.com/jaimergp/miniforge/releases/download/24.11.2-1_colab/Miniforge3-colab-24.11.2-1_colab-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:11
🔁 Restarting kernel...


In [ ]:
# update conda env with the SV.yaml file
!conda env update -n base -f /content/Nanopore_Sequencing_Lab/SV.yaml

Channels:
 - bioconda
 - conda-forge
Platform: linux-64
Solving environment: - \ | / - \ | / - done


==> WARNING: A newer version of conda exists. <==
    current version: 24.11.3
    latest version: 26.5.3

Please update conda by running

    $ conda update -n base -c conda-forge conda


#
# To activate this environment, use
#
#     $ conda activate base
#
# To deactivate an active environment, use
#
#     $ conda deactivate



You are a **health researcher** in a hospital. You have to analyze the biological sample of a patient with a suspected neurological disorders.
Your task is to **analyze** the FGF14 gene on chromosome 13 in their **sequencing data**. **Identify any mutations** that could explain the condition and **lead to a diagnosis**. Use bioinformatics tools to detect potential pathogenic variants.

In [ ]:
#%%bash
# download the human reference genome (T2T, chr13) and the sequencing data (.fastq format) from Zenodo repository
!wget https://zenodo.org/records/20795528/files/chr13_T2T.fasta
!wget https://zenodo.org/records/20795528/files/chr13_hg38.fa
!wget https://zenodo.org/records/20795528/files/FGF14.fastq
!wget https://zenodo.org/records/20795528/files/variant_catalog_hg38.json
!wget https://zenodo.org/records/20795528/files/snp.zip

--2026-06-22 08:36:07--  https://zenodo.org/records/20759671/files/chr13_T2T.fasta
Resolving zenodo.org (zenodo.org)... 188.184.98.114, 137.138.52.235, 137.138.153.219, ...
Connecting to zenodo.org (zenodo.org)|188.184.98.114|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 115838027 (110M) [application/octet-stream]
Saving to: ‘chr13_T2T.fasta’

chr13_T2T.fasta     100%[===================>] 110.47M  3.41MB/s    in 92s     

2026-06-22 08:37:40 (1.21 MB/s) - ‘chr13_T2T.fasta’ saved [115838027/115838027]

--2026-06-22 08:37:40--  https://zenodo.org/records/20759671/files/FGF14.fastq
Resolving zenodo.org (zenodo.org)... 137.138.52.235, 188.185.48.75, 188.184.98.114, ...
Connecting to zenodo.org (zenodo.org)|137.138.52.235|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 13667030 (13M) [application/octet-stream]
Saving to: ‘FGF14.fastq’

FGF14.fastq         100%[===================>]  13.03M   965KB/s    in 13s     

2026-06-22 08:37:53 (

In [ ]:
%%bash

# Look at .fastq and .fasta file
echo "First 4 lines of the .fastq file"

head -n 4 /content/Nanopore_Sequencing_Lab/FGF14.fastq

echo "First 10 lines of the reference sequence"

head -n 10 /content/Nanopore_Sequencing_Lab/chr13_T2T.fasta

First 4 lines of the .fastq file \n
@d5c334b3-eba5-4b53-b264-18bc2e7c57e5
...
+
GHFSLSSSLSSSOINSSHGDADSDELMMSSSJLOSJHSS2-,)+,,,,0....;BC?DEKD>?>>;FS?=64556DSKHSSLKSSSSIFEDEB/&&&&&;FGFSSNSSSSSSSSOSSSSSSSPOSSSSKKSSSMKIG7SEBA@>>?@H>>KSLJKSSSOIJEDS?SSSSSSSSSSSSSSSSKHNPEH>?>?ALSSGSSJOSSSHI=@ABDEGGA??9879?>@C=<;?8?KSMSSSRSSSSSSSNIHFE22222BIKIHLLSSMSJLSSSNKOMB>=9:GHJSSSSPSSSSPSSSSMSSSRJORSMSSSNKKMMSSLMMJSKKHLSLNSSSSSSSIGFCEB4>;-5GHJSSSSSSSSPSMLKSSPSSSSSSSSSSMSJPGHKIID<>@GEGFE?;<<=A>==;6/0CCSSSKJSPSNPSSMSSSSSSSSSPSSSSSSSSSSOMLIJLSSKJLSSSKGGES7..---2222>GGFJHHLSLSJMSLSSSSIIFFESOSSSSSSSSSNSSSSSSSSSMNLMSRKSSSKSOJKHLSOISSSPSLLSSSSLLLJPSSSNSSGESGDDGCSA>9999;=....../0014/.///21244@CDD?>>>CBGSSSHKSJJMSKNSSORSSSSSSSSSSSILMJSSSLISSSSNSSNSSSSSSLSJSSSLHNSSSNSSSSOSSSSSSSSSSLHSSSHHHF9:<99<OSHKSIKSSSOLLLLILOQMKNPMKSQBAA@?<?@@@@A@AASMSSSOSSSSILSSSJSSOSSSSJKDB<2-,++,6@7@CCC::::988888E>:::EDNQSSSSSMNKSOLOJB9732224:?DGEFB;;::;==>44///133223GIGJIICCGC>4449<::46>94@AAA=:888:DSSSSSSSJHLJHE:::::;EIPJ?<<==NJIKHSSSSS

In [ ]:
# Creating the reference genome index file (.fasta.fai)

!samtools faidx /content/Nanopore_Sequencing_Lab/chr13_T2T.fasta
!samtools faidx /content/Nanopore_Sequencing_Lab/chr13_hg38.fasta

In [ ]:
%%bash
# 1. Mapping the sequencing data to the reference sequence (chr13, T2T human genome) with minimap2 -> .sam file

minimap2 -ax map-ont -t 1 /content/Nanopore_Sequencing_Lab/chr13_hg38.fa /content/Nanopore_Sequencing_Lab/FGF14.fastq > /content/Nanopore_Sequencing_Lab/FGF14.sam

echo "minimap2 alignment to the reference genome succesfully completed"

minimap2 alignment to the reference genome succesfully completed


[M::mm_idx_gen::7.111*0.98] collected minimizers
[M::mm_idx_gen::9.147*0.99] sorted minimizers
[M::main::9.148*0.99] loaded/built the index for 1 target sequence(s)
[M::mm_mapopt_update::9.409*0.98] mid_occ = 151
[M::mm_idx_stat] kmer size: 15; skip: 10; is_hpc: 0; #seq: 1
[M::mm_idx_stat::9.569*0.98] distinct minimizers: 12510280 (82.06% are singletons); average occurrences: 1.474; average spacing: 6.204; total length: 114364328
[M::worker_pipeline::14.881*0.98] mapped 1992 sequences
[M::main] Version: 2.31-r1302
[M::main] CMD: minimap2 -ax map-ont -t 1 /content/chr13_hg38.fa /content/Nanopore_Sequencing_Lab/FGF14.fastq
[M::main] Real time: 14.916 sec; CPU: 14.670 sec; Peak RSS: 0.941 GB


In [ ]:
%%bash
# 2. Samtools post processing: coversion to bam file, sorting and indexing (.bam.bai)

samtools view -S -b /content/Nanopore_Sequencing_Lab/FGF14.sam | samtools sort -o /content/Nanopore_Sequencing_Lab/FGF14.sorted.bam
samtools index /content/Nanopore_Sequencing_Lab/FGF14.sorted.bam

echo "samtools post processing succesfully completed"

samtools post processing succesfully completed


In [ ]:
%%bash
echo "First 4 lines of the .fastq file"




First 4 lines of the .fastq file


In [ ]:
%%bash
# 2.1 Estimate average coverage using samtools depth

echo "Average coverage of the sample"

samtools depth /content/Nanopore_Sequencing_Lab/FGF14.sorted.bam | awk '{sum+=$3} END {print sum/NR}'

Average coverage of the sample
24.6404


In [ ]:
%%bash

# 3. Structural Variants calling with sniffles

sniffles -i /content/Nanopore_Sequencing_Lab/FGF14.sorted.bam -v /content/Nanopore_Sequencing_Lab/FGF14_SV.vcf --threads 4 --allow-overwrite --no-qc

2026-06-22 10:01:00,402 INFO sniffles.main (25016): Running Sniffles2, build 2.8.0
2026-06-22 10:01:00,402 INFO sniffles.main (25016):   Run Mode: call_sample
2026-06-22 10:01:00,402 INFO sniffles.main (25016):   Start on: 2026/06/22 10:01:00
2026-06-22 10:01:00,402 INFO sniffles.main (25016):   Working dir: /content/Nanopore_Sequencing_Lab
2026-06-22 10:01:00,402 INFO sniffles.main (25016):   Used command: /usr/local/bin/sniffles -i /content/Nanopore_Sequencing_Lab/FGF14.sorted.bam -v /content/Nanopore_Sequencing_Lab/FGF14_SV.vcf --threads 4 --allow-overwrite --no-qc
2026-06-22 10:01:00,402 INFO sniffles.main (25016): ==============================
2026-06-22 10:01:00,426 INFO sniffles.main (25016): Opening for reading: /content/Nanopore_Sequencing_Lab/FGF14.sorted.bam
2026-06-22 10:01:00,428 INFO sniffles.main (25016): Opening for writing: /content/Nanopore_Sequencing_Lab/FGF14_SV.vcf (single-sample, sorted)
2026-06-22 10:01:00,435 INFO sniffles.main (25016): 
2026-06-22 10:01:00,435

In [ ]:
%%bash
# Look into .vcf file

cat /content/Nanopore_Sequencing_Lab/FGF14_SV.vcf

##fileformat=VCFv4.2
##source=Sniffles2_2.8.0
##command="/usr/local/bin/sniffles -i /content/Nanopore_Sequencing_Lab/FGF14.sorted.bam -v /content/Nanopore_Sequencing_Lab/FGF14_SV.vcf --threads 4 --allow-overwrite --no-qc"
##fileDate="2026/06/22 10:01:00"
##contig=<ID=chr13,length=114364328>
##ALT=<ID=INS,Description="Insertion">
##ALT=<ID=DEL,Description="Deletion">
##ALT=<ID=DUP,Description="Duplication">
##ALT=<ID=INV,Description="Inversion">
##ALT=<ID=BND,Description="Breakend; Translocation">
##FORMAT=<ID=GT,Number=1,Type=String,Description="Genotype">
##FORMAT=<ID=GQ,Number=1,Type=Integer,Description="Genotype quality">
##FORMAT=<ID=DR,Number=1,Type=Integer,Description="Number of reference reads">
##FORMAT=<ID=DV,Number=1,Type=Integer,Description="Number of variant reads">
##FORMAT=<ID=PS,Number=1,Type=Integer,Description="Phase-block, zero if none or not phased">
##FORMAT=<ID=ID,Number=1,Type=String,Description="Individual sample SV ID for multi-sample output">
##FILTER=<ID=PASS,

In [ ]:
%%bash

# Download pytorch model for SNP variant calling

cd /usr/local/bin/
rm -r ./models
mkdir models
cd ./models
wget -r -np -nH --cut-dirs=3 -R "index.html*" https://www.bio8.cs.hku.hk/clair3/clair3_models_pytorch/r1041_e82_400bps_sup_v520_with_mv

--2026-06-22 09:07:02--  https://www.bio8.cs.hku.hk/clair3/clair3_models_pytorch/r1041_e82_400bps_sup_v520_with_mv
Resolving www.bio8.cs.hku.hk (www.bio8.cs.hku.hk)... 147.8.177.118
Connecting to www.bio8.cs.hku.hk (www.bio8.cs.hku.hk)|147.8.177.118|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://www.bio8.cs.hku.hk/clair3/clair3_models_pytorch/r1041_e82_400bps_sup_v520_with_mv/ [following]
--2026-06-22 09:07:03--  https://www.bio8.cs.hku.hk/clair3/clair3_models_pytorch/r1041_e82_400bps_sup_v520_with_mv/
Reusing existing connection to www.bio8.cs.hku.hk:443.
HTTP request sent, awaiting response... 200 OK
Length: 1303 (1.3K) [text/html]
Saving to: ‘r1041_e82_400bps_sup_v520_with_mv’

     0K .                                                     100%  309M=0s

2026-06-22 09:07:03 (309 MB/s) - ‘r1041_e82_400bps_sup_v520_with_mv’ saved [1303/1303]

Loading robots.txt; please ignore errors.
--2026-06-22 09:07:03--  https://www.bio8.cs.hku.hk

In [ ]:
%%bash


UsageError: %%bash is a cell magic, but the cell body is empty.


In [ ]:
%%bash

# 4. SNPs calling using clair3

run_clair3.sh -b /content/Nanopore_Sequencing_Lab/FGF14.sorted.bam -f /content/Nanopore_Sequencing_Lab/chr13_T2T.fasta -p ont -t 2 -o clair3_out -m /usr/local/bin/models/r1041_e82_400bps_sup_v520_with_mv/

[WARNING] No absolute output path provided, using current directory as prefix
[ERROR] Conda prefix not found, please activate clair3 conda environment first, model path: /usr/local/bin/models/r1041_e82_400bps_sup_v520_with_mv


CalledProcessError: Command 'b'\n# 4. SNPs calling using clair3\n\nrun_clair3.sh -b /content/Nanopore_Sequencing_Lab/FGF14.sorted.bam -f /content/Nanopore_Sequencing_Lab/chr13_T2T.fasta -p ont -t 2 -o clair3_out -m /usr/local/bin/models/r1041_e82_400bps_sup_v520_with_mv/\n'' returned non-zero exit status 1.

In [ ]:
%%bash
# Look into .vcf file of clair3

cat /content/Nanopore_Sequencing_Lab/clair3_out/pileup.vcf.gz

In [ ]:
%%bash

# clone the straglr repository (software for STRs calling)

cd /content/Nanopore_Sequencing_Lab/
git clone https://github.com/BirolLab/straglr
cd /content/Nanopore_Sequencing_Lab/straglr

Cloning into 'straglr'...


In [ ]:
%%bash

# 5. Accurate STRs detection with straglr

python /content/Nanopore_Sequencing_Lab/straglr/straglr.py /content/Nanopore_Sequencing_Lab/FGF14.sorted.bam /content/Nanopore_Sequencing_Lab/chr13_hg38.fa FGF14_STR --min_str_len 2 --min_support 5 --max_str_len 100000 --max_num_clusters 2 --min_cluster_d 5

In [ ]:
%%bash

head -n 40 /content/Nanopore_Sequencing_Lab/FGF14_STR.tsv

#2026-06-22_11:08:15 /content/Nanopore_Sequencing_Lab/straglr/straglr.py /content/Nanopore_Sequencing_Lab/FGF14.sorted.bam /content/chr13_hg38.fa FGF14_STR --min_str_len 2 --min_support 5 --max_str_len 100000 --max_num_clusters 2 --min_cluster_d 5
#chrom	start	end	target_repeat	locus	coverage	genotype	read_name	actual_repeat	copy_number	size	read_start	strand	allele	read_status
chr13	102161566	102161726	AAG	chr13:102161566-102161726	34	250.2(10);171.0(12)	eee0d64a-23e6-45a8-92bd-7015dc5c761f	AAG	255.0	765	20661	-	250.2	full
chr13	102161566	102161726	AAG	chr13:102161566-102161726	34	250.2(10);171.0(12)	6d11b152-22ec-4bf6-89c3-1e10b92735a1	AAG	252.0	756	36435	-	250.2	full
chr13	102161566	102161726	AAG	chr13:102161566-102161726	34	250.2(10);171.0(12)	fedacbd2-cb57-479d-a7cd-58c2d39d9cfe	AAG	251.7	755	4674	-	250.2	full
chr13	102161566	102161726	AAG	chr13:102161566-102161726	34	250.2(10);171.0(12)	1e9b3d93-aa6b-4e81-8050-06001bc1c8fd	AAG	251.3	754	9633	-	250.2	full
chr13	102161566	102161726

In [ ]:
%%bash

# 6. Phasing and genotyping of the mapped reads using clair3 output and whatshap

whatshap phase --reference /content/Nanopore_Sequencing_Lab/chr13_T2T.fasta -o /content/Nanopore_Sequencing_Lab/FGF14.phased.vcf.gz /content/Nanopore_Sequencing_Lab/clair3_out/pileup.vcf.gz /content/Nanopore_Sequencing_Lab/FGF14.sorted.bam --ignore-read-groups
tabix -p vcf /content/Nanopore_Sequencing_Lab/FGF14.phased.vcf.gz
whatshap haplotag -o /content/Nanopore_Sequencing_Lab/FGF14.phased.bam --reference /content/Nanopore_Sequencing_Lab/chr13_T2T.fasta /content/Nanopore_Sequencing_Lab/FGF14.phased.vcf.gz /content/Nanopore_Sequencing_Lab/FGF14.sorted.bam --ignore-read-groups
samtools index /content/Nanopore_Sequencing_Lab/FGF14.phased.bam

In [ ]:
%%bash

# 7. Using bcftools to make a faster SNP calling and query

bcftools mpileup -Ou -f /content/Nanopore_Sequencing_Lab/chr13_hg38.fa /content/Nanopore_Sequencing_Lab/FGF14.sorted.bam 2> /dev/null | bcftools call -mv -Ov -o /content/Nanopore_Sequencing_Lab/FGF14_bcftools.vcf


Note: none of --samples-file, --ploidy or --ploidy-file given, assuming all sites are diploid


In [ ]:
%%bash

bcftools query -f '%POS\t%REF\t%ALT\t%INFO/DP4\n' /content/Nanopore_Sequencing_Lab/FGF14_bcftools.vcf

46217204	A	C	0,0,0,1
46217229	T	C	0,0,0,1
46217244	T	C	0,0,0,1
46217247	C	T	0,0,0,1
46217255	G	T	0,0,0,1
46217273	G	T	0,0,0,1
46217287	A	G	0,0,0,1
46217289	T	C	0,0,0,1
46217302	G	A	0,0,0,1
46217308	G	A	0,0,0,1
46217311	C	T	0,0,0,1
46217314	G	A	0,0,0,1
46217315	A	C	0,0,0,1
46217330	C	A	0,0,0,1
46217331	A	G	0,0,0,1
46217350	T	G	0,0,0,1
46217357	A	G	0,0,0,1
46217372	T	C	0,0,0,1
46217423	G	A	0,0,0,1
46217456	C	A	0,0,0,1
46217469	G	C	0,0,0,1
46217484	A	G	0,0,0,1
46217514	A	T	0,0,0,1
46217540	C	A	0,0,0,1
46217541	T	C	0,0,0,1
46217552	A	C	0,0,0,1
46217554	C	T	0,0,0,1
46217563	G	A	0,0,0,1
46217564	G	A	0,0,0,1
46217568	C	A	0,0,0,1
46217573	A	T	0,0,0,1
46217574	T	G	0,0,0,1
46217581	A	G	0,0,0,1
46217582	A	G	0,0,0,1
46217592	A	T	0,0,0,1
46217650	T	G	0,0,0,1
46217701	G	C	0,0,0,1
46217702	G	A	0,0,0,1
46217707	A	T	0,0,0,1
46217714	C	G	0,0,0,1
46217736	G	T	0,0,0,1
46217756	T	A	0,0,0,1
46217758	C	G	0,0,0,1
46217760	C	G	0,0,0,1
46217765	C	A	0,0,0,1
46217777	A	G	0,0,0,1
46217778	A	C	0,0,0,1
46217786	A	C	

[W::bcf_hdr_check_sanity] MQ should be declared as Type=Float


In [ ]:
%%bash

# 8. SV annotation with VEP

vep -i /content/Nanopore_Sequencing_Lab/FGF14_SV.vcf -o /content/Nanopore_Sequencing_Lab/FGF14_SV_annotated.vcf --vcf --assembly GRCh38 --symbol --everything


Download AnnotSV supporting data files:


Download Exomiser supporting data files:

Archive:  _phenotype.zip
Installation completed. Annotation files have been generated in ./AnnotSV_annotations.


/usr/local/bin/INSTALL_annotations.sh: line 48: [: missing `]'
mkdir: cannot create directory ‘./AnnotSV_annotations’: File exists
  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed
100     63 100     63   0      0    129      0                              0
tar: This does not look like a tar archive

gzip: stdin: not in gzip format
tar: Child returned status 1
tar: Error is not recoverable: exiting now
  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed
100    162 100    162   0      0   1811      0                              0
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive

In [ ]:
%%bash
snpEff download GRCh38.99

In [ ]:
%%bash
# 9. SNPs annotation with snpEff

snpEff -Xmx8g -noStats GRCh38.99 /content/Nanopore_Sequencing_Lab/FGF14_bcftools.vcf > /content/Nanopore_Sequencing_Lab/FGF14_bcftools_annotated.vcf

In [ ]:
%%bash
head -n 40 /content/Nanopore_Sequencing_Lab/FGF14_bcftools_annotated.vcf

##fileformat=VCFv4.2
##FILTER=<ID=PASS,Description="All filters passed">
##bcftoolsVersion=1.23.1+htslib-1.23.1
##bcftoolsCommand=mpileup -Ou -f /content/chr13_hg38.fa /content/Nanopore_Sequencing_Lab/FGF14.sorted.bam
##reference=file:///content/chr13_hg38.fa
##contig=<ID=chr13,length=114364328>
##ALT=<ID=*,Description="Represents allele(s) other than observed.">
##INFO=<ID=INDEL,Number=0,Type=Flag,Description="Indicates that the variant is an INDEL.">
##INFO=<ID=IDV,Number=1,Type=Integer,Description="Maximum number of raw reads supporting an indel">
##INFO=<ID=IMF,Number=1,Type=Float,Description="Maximum fraction of raw reads supporting an indel">
##INFO=<ID=DP,Number=1,Type=Integer,Description="Raw read depth">
##INFO=<ID=VDB,Number=1,Type=Float,Description="Variant Distance Bias for filtering splice-site artefacts in RNA-seq data (bigger is better)",Version="3">
##INFO=<ID=RPBZ,Number=1,Type=Float,Description="Mann-Whitney U-z test of Read Position Bias (closer to 0 is better)">
##I

In [ ]:
%%bash
# 10. STRs annotation with stranger
stranger -f /content/variant_catalog_hg38.json /content/Nanopore_Sequencing_Lab/FGF14_STR.vcf > /content/Nanopore_Sequencing_Lab/FGF14_STR_annotated.vcf

2026-06-22 11:15:57 c494abda4517 stranger.cli[43953] INFO Running stranger version 0.10.2
2026-06-22 11:15:57 c494abda4517 stranger.cli[43953] INFO Parsing repeats file /content/variant_catalog_hg38.json
2026-06-22 11:15:58 c494abda4517 stranger.utils[43953] WARNING No info for repeat id None


In [ ]:
%%bash
head -n 40 /content/Nanopore_Sequencing_Lab/FGF14_STR_annotated.vcf

##fileformat=VCFv4.5
##fileDate=20260622
##source=StraglrV1.5.6
##reference=/content/chr13_hg38.fa
##contig=<ID=chr13,length=114364328>
##INFO=<ID=END,Number=1,Type=Integer,Description="End position of the variant described in this record">
##INFO=<ID=LOCUS,Number=1,Type=String,Description="Locus ID">
##INFO=<ID=RUS_REF,Number=1,Type=String,Description="Repeat unit sequence in the reference sequence">
##INFO=<ID=RUL_REF,Number=1,Type=String,Description="Repeat unit length in the reference sequence">
##INFO=<ID=SVLEN,Number=A,Type=Integer,Description="Length of structural variant">
##INFO=<ID=RN,Number=A,Type=Integer,Description="Total number of repeat sequences in this allele">
##INFO=<ID=RUS,Number=.,Type=String,Description="Repeat unit sequence of the corresponding repeat sequence">
##INFO=<ID=RUL,Number=.,Type=Integer,Description="Repeat unit length of the corresponding repeat sequence">
##INFO=<ID=RUC,Number=.,Type=Float,Description="Repeat unit count of corresponding repeat sequen